# Robots Dynamic: автономный pipeline

Этот notebook выполняет полный сценарий:

1. загрузка фактических данных;
2. обучение one-step GNN;
3. обучение Macrostate KMeans на fact-data;
4. multistep evaluation на fact-data;
5. генерация синтетических начальных условий;
6. synthetic rollout на 600 шагов;
7. анализ формирования мицеллы.

Единственный обязательный внешний файл: `data/processed_robots_data.parquet`. Веса GNN обучаются и сохраняются внутри этой папки.

In [ ]:
from pathlib import Path

from config import ResearchConfig
from experiment_runner import ResearchPipeline
from initial_state_generator import InitialConditionSpec, InitialStateGenerator

# Центральная конфигурация
config = ResearchConfig(
    data_path="data/processed_robots_data.parquet",
    model_path="models/gnn_model_weights.pth",
    output_dir="results/research",

    # GNN
    hidden_dim=128,
    gnn_layers=2,
    dropout=0.1,
    k_neighbors=5,

    # Обучение one-step
    train_one_step=True,
    skip_training_if_model_exists=True,  # поставьте False или force=True ниже для переобучения
    batch_size=512,
    num_epochs=50,
    learning_rate=1e-3,
    weight_decay=1e-4,
    early_stopping_patience=10,
    train_sample_ratio=1.0,  # доля fact-файлов для обучения: 1.0 = все, 0.5 = 50%
    max_train_samples_per_file=None,  # можно поставить число для ускоренного debug

    # Multistep / synthetic
    start_step=3,
    synthetic_total_steps=600,
    n_robots=50,
    dt=0.1,

    # Macrostate
    n_clusters=4,
    robot_size=60.0,
    random_state=42,

    # Видео/анимации: создаются отдельными ячейками после обучения/evaluation
    save_animations=True,
    auto_save_animations_during_pipeline=False,  # видео НЕ создаётся внутри train/evaluate
    animation_fps=10,
    animation_max_frames=300,  # None или -1 в CLI для всех кадров

    # Демонстрационный эксперимент для графиков/видео.
    # Логика как в GitHub main.ipynb: выбрать конкретный test-файл, например test_files[2].
    demo_split="test",
    demo_file_index=2,
    demo_file_name=None,  # можно явно указать: "01_69_[45_bots_PWM_2_ex_108].pickle"
)

config.ensure_dirs()
pipeline = ResearchPipeline(config)
print(config)

## 1. Загрузка fact-data

Данные должны лежать в `data/processed_robots_data.parquet`.

In [ ]:
experiments = pipeline.load_fact_data()
print(f"Loaded experiments: {len(experiments)}")
first_name = next(iter(experiments))
print("First experiment:", first_name, experiments[first_name]["tensor"].shape)

## 1.1. Выбор демонстрационного эксперимента

Сохранена логика GitHub notebook: выбирается конкретный эксперимент из test split, по умолчанию `test_files[2]`. Этот `demo_file` используется для графиков ошибок и видео факт/прогноз.


In [ ]:
demo_file = pipeline.select_demo_file(experiments)
demo_file


## 2. Обучение one-step GNN

Модель обучается внутри автономной папки и сохраняется в `models/gnn_model_weights.pth`.

Если файл уже существует и `skip_training_if_model_exists=True`, обучение будет пропущено. Для принудительного переобучения поставьте `force=True`.

In [ ]:
one_step_metrics = pipeline.train_one_step_model(experiments, force=False)
one_step_metrics

## 2.1. Видео one-step после обучения

Эта ячейка не переобучает модель. Она читает сохранённый `results/gnn_predictions.parquet`, фильтрует строки по `demo_file` — аналогично `sub_df = df[df['file_name'] == test_files[2]]` из GitHub — и создаёт видео только для выбранного эксперимента.


In [ ]:
one_step_video_path = pipeline.save_one_step_video(file_name=demo_file)
one_step_video_path


## 3. Обучение Macrostate KMeans на фактических данных

Кластеры/состояния системы берутся из fact-data. Синтетические траектории дальше только классифицируются этим KMeans.

In [ ]:
macro_classifier = pipeline.fit_macrostate_classifier(experiments)
macro_classifier.interpretation_df

## 4. Multistep evaluation на fact-data

Проверяем качество autoregressive-прогноза: после seed-history `0..3` модель не использует фактические шаги, а прогнозирует сама себя вперед.

In [ ]:
pipeline.load_predictor(train_if_missing=False)
multistep_predictions, multistep_metrics = pipeline.evaluate_multistep_on_fact_data(
    experiments,
    horizon=200,  # можно None для всего эксперимента
)
multistep_metrics

In [ ]:
multistep_predictions.head()

## 4.1. Видео multistep после evaluation

Эта ячейка не запускает rollout повторно. Она использует `multistep_predictions` и фильтрует их по тому же `demo_file`, чтобы видео соответствовало выбранному эксперименту.


In [ ]:
multistep_video_path = pipeline.save_multistep_video(
    predictions_df=multistep_predictions,
    file_name=demo_file,
)
multistep_video_path


## 5. Генерация сетки синтетических начальных условий

Можно менять радиусы, скорости, режимы скоростей и число повторов.

In [ ]:
base_spec = InitialConditionSpec(
    experiment_id="base",
    n_robots=config.n_robots,
    history_len=config.start_step + 1,
    dt=config.dt,
    dish_diameter=config.dish_diameter,
    robot_diameter=config.robot_size,
    min_center_distance=config.min_center_distance,
    seed=config.random_state,
)

specs = InitialStateGenerator.make_sweep_specs(
    base_spec,
    # For a dish diameter of 1000 and robot diameter of 60, centers can use max radius 470.
    # These values represent dense / medium / sparse initial systems.
    radius_values=[250, 350, 470],
    speed_values=None,  # None => sample continuous random speeds from speed_range
    speed_range=(config.speed_min, config.speed_max),
    n_speed_samples=config.n_speed_samples,  # currently 3 sampled speed variants
    speed_seed=config.random_state,
    velocity_modes=["random", "aligned", "rotational", "inward"],
    repeats=2,
)

print(f"Sampled speed_mean values: {sorted({round(s.speed_mean, 3) for s in specs})}")
print(f"Synthetic experiments: {len(specs)}")
specs[:3]


## 6. Synthetic study на 600 шагов

Для каждого синтетического эксперимента:

1. генерируется initial history длины `start_step+1`;
2. запускается multistep rollout на 600 шагов;
3. каждый шаг классифицируется Macrostate KMeans;
4. считается факт появления мицеллы, первый шаг и длительность.

In [ ]:
states_df, summary_df = pipeline.run_synthetic_study(
    specs,
    total_steps=config.synthetic_total_steps,
    save_trajectories=False,
)

summary_df.head()

In [ ]:
states_df.head()

## 7. Полностью автоматический запуск одной командой

Эта ячейка делает всё сразу: one-step train/reuse → KMeans → multistep evaluation → synthetic study.

Она дублирует этапы выше, поэтому обычно её запускают вместо отдельных ячеек.

In [ ]:
# result = pipeline.run_full_research(
#     specs,
#     experiments_dict=experiments,
#     train_one_step=True,
#     force_retrain=False,
#     evaluate_real_horizon=200,
#     synthetic_total_steps=600,
#     save_trajectories=False,
# )
# result["synthetic_summary"].head()

## Выходные файлы

Основные результаты сохраняются в:

```text
models/gnn_model_weights.pth
results/research/tables/one_step_training_history.csv
results/research/tables/one_step_test_metrics.csv
results/research/tables/fact_cluster_results.csv
results/research/tables/multistep_fact_metrics.csv
results/research/tables/synthetic_step_states.csv
results/research/tables/synthetic_summary.csv
results/research/plots/*.png
```